In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(r"c:/Users/ADMIN/Documents/Datathon/datathon-2026-round-1")

orders = pd.read_csv(DATA_DIR / "orders.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")
web_traffic = pd.read_csv(DATA_DIR / "web_traffic.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv", low_memory=False)
customers = pd.read_csv(DATA_DIR / "customers.csv")
geography = pd.read_csv(DATA_DIR / "geography.csv")
payments = pd.read_csv(DATA_DIR / "payments.csv")

for col in ["order_date"]:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

if "date" in web_traffic.columns:
    web_traffic["date"] = pd.to_datetime(web_traffic["date"], errors="coerce")

print("Loaded:", {
    "orders": orders.shape,
    "products": products.shape,
    "returns": returns.shape,
    "web_traffic": web_traffic.shape,
    "order_items": order_items.shape,
    "customers": customers.shape,
    "geography": geography.shape,
    "payments": payments.shape,
})

Loaded: {'orders': (646945, 8), 'products': (2412, 8), 'returns': (39939, 7), 'web_traffic': (3652, 7), 'order_items': (714669, 7), 'customers': (121930, 7), 'geography': (39948, 4), 'payments': (646945, 4)}


In [2]:
# Q1: Median inter-order gap (days) for customers with >1 order
q1 = orders[["customer_id", "order_date"]].dropna().copy()
q1 = q1.sort_values(["customer_id", "order_date"])
q1["gap_days"] = q1.groupby("customer_id")["order_date"].diff().dt.days
multi_customer = q1.groupby("customer_id")["order_date"].transform("size") > 1
median_gap = q1.loc[multi_customer & q1["gap_days"].notna(), "gap_days"].median()

choices_q1 = {"A": 30, "B": 90, "C": 144, "D": 365}
ans_q1 = min(choices_q1, key=lambda k: abs(choices_q1[k] - median_gap))

print(f"Q1 median inter-order gap: {median_gap:.2f} days")
print(f"Closest option: {ans_q1}) {choices_q1[ans_q1]} days")

Q1 median inter-order gap: 144.00 days
Closest option: C) 144 days


In [3]:
# Q2: Segment with highest average gross margin (price - cogs) / price
q2 = products[["segment", "price", "cogs"]].copy()
q2 = q2[q2["price"] > 0]
q2["gross_margin"] = (q2["price"] - q2["cogs"]) / q2["price"]
segment_margin = q2.groupby("segment", dropna=True)["gross_margin"].mean().sort_values(ascending=False)

top_segment = segment_margin.index[0]
choices_q2 = {"A": "Premium", "B": "Performance", "C": "Activewear", "D": "Standard"}
ans_q2 = next((k for k, v in choices_q2.items() if v == top_segment), "N/A")

print("Q2 average gross margin by segment:")
print(segment_margin)
print(f"Top segment: {top_segment}")
print(f"Option: {ans_q2}) {top_segment}")

Q2 average gross margin by segment:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64
Top segment: Standard
Option: D) Standard


In [4]:
# Q3: Most common return reason for Streetwear products
q3 = returns.merge(products[["product_id", "category"]], on="product_id", how="left")
q3 = q3[q3["category"] == "Streetwear"]
reason_counts = q3["return_reason"].value_counts(dropna=False)

top_reason = reason_counts.index[0]
choices_q3 = {"A": "defective", "B": "wrong_size", "C": "changed_mind", "D": "not_as_described"}
ans_q3 = next((k for k, v in choices_q3.items() if v == top_reason), "N/A")

print("Q3 return reason counts (Streetwear):")
print(reason_counts)
print(f"Top reason: {top_reason}")
print(f"Option: {ans_q3}) {top_reason}")

Q3 return reason counts (Streetwear):
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
Top reason: wrong_size
Option: B) wrong_size


In [5]:
# Q4: Traffic source with lowest average bounce_rate
q4 = web_traffic.dropna(subset=["traffic_source", "bounce_rate"]).copy()
source_bounce = q4.groupby("traffic_source")["bounce_rate"].mean().sort_values()

top_source = source_bounce.index[0]
choices_q4 = {"A": "organic_search", "B": "paid_search", "C": "email_campaign", "D": "social_media"}
ans_q4 = next((k for k, v in choices_q4.items() if v == top_source), "N/A")

print("Q4 mean bounce_rate by source:")
print(source_bounce)
print(f"Lowest source: {top_source}")
print(f"Option: {ans_q4}) {top_source}")

Q4 mean bounce_rate by source:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
Lowest source: email_campaign
Option: C) email_campaign


In [6]:
# Q5: Percentage of order_items rows with promotion applied (promo_id not null)
q5 = order_items.copy()
promo_applied = q5["promo_id"].notna() & q5["promo_id"].astype(str).str.strip().ne("") & q5["promo_id"].astype(str).str.lower().ne("nan")
promo_pct = promo_applied.mean() * 100

choices_q5 = {"A": 12, "B": 25, "C": 39, "D": 54}
ans_q5 = min(choices_q5, key=lambda k: abs(choices_q5[k] - promo_pct))

print(f"Q5 promo-applied percentage: {promo_pct:.2f}%")
print(f"Closest option: {ans_q5}) {choices_q5[ans_q5]}%")

Q5 promo-applied percentage: 38.66%
Closest option: C) 39%


In [7]:
# Q6: Age group with highest avg orders per customer (age_group not null)
q6_customers = customers.loc[customers["age_group"].notna(), ["customer_id", "age_group"]].copy()
order_count_per_customer = orders.groupby("customer_id").size().rename("num_orders")
q6 = q6_customers.merge(order_count_per_customer, on="customer_id", how="left")
q6["num_orders"] = q6["num_orders"].fillna(0)
age_group_avg_orders = q6.groupby("age_group")["num_orders"].mean().sort_values(ascending=False)

top_age_group = age_group_avg_orders.index[0]
choices_q6 = {"A": "55+", "B": "25-34", "C": "35-44", "D": "45-54"}
ans_q6 = next((k for k, v in choices_q6.items() if v == top_age_group), "N/A")

print("Q6 avg orders/customer by age_group:")
print(age_group_avg_orders)
print(f"Top age group: {top_age_group}")
print(f"Option: {ans_q6}) {top_age_group}")

Q6 avg orders/customer by age_group:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: num_orders, dtype: float64
Top age group: 55+
Option: A) 55+


In [8]:
# Q7: Region with highest total revenue
# Note: The prompt mentions sales_train.csv, but this workspace has sales.csv only (no region key).
# So we compute regional revenue from order_items + orders(zip) + geography(region).
geo_region = geography[["zip", "region"]].drop_duplicates(subset=["zip"])
q7 = order_items.merge(orders[["order_id", "zip"]], on="order_id", how="left")
q7 = q7.merge(geo_region, on="zip", how="left")
q7["line_revenue"] = q7["quantity"] * q7["unit_price"] - q7["discount_amount"]
region_revenue = q7.groupby("region", dropna=True)["line_revenue"].sum().sort_values(ascending=False)

top_region = region_revenue.index[0]
choices_q7 = {"A": "West", "B": "Central", "C": "East", "D": "Cả ba vùng có doanh thu xấp xỉ bằng nhau"}
ans_q7 = next((k for k, v in choices_q7.items() if v == top_region), "N/A")

print("Q7 total revenue by region:")
print(region_revenue)
print(f"Top region: {top_region}")
print(f"Option: {ans_q7}) {top_region}")

Q7 total revenue by region:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: line_revenue, dtype: float64
Top region: East
Option: C) East


In [9]:
# Q8: Most used payment_method among cancelled orders
q8 = orders[orders["order_status"] == "cancelled"]
payment_counts_cancelled = q8["payment_method"].value_counts(dropna=False)

top_payment_cancelled = payment_counts_cancelled.index[0]
choices_q8 = {"A": "credit_card", "B": "cod", "C": "paypal", "D": "bank_transfer"}
ans_q8 = next((k for k, v in choices_q8.items() if v == top_payment_cancelled), "N/A")

print("Q8 cancelled-order payment method counts:")
print(payment_counts_cancelled)
print(f"Most used: {top_payment_cancelled}")
print(f"Option: {ans_q8}) {top_payment_cancelled}")

Q8 cancelled-order payment method counts:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
Most used: credit_card
Option: A) credit_card


In [10]:
# Q9: Size with highest return rate = (#returns records) / (#order_items rows)
size_choices = ["S", "M", "L", "XL"]

order_items_size = order_items.merge(products[["product_id", "size"]], on="product_id", how="left")
returns_size = returns.merge(products[["product_id", "size"]], on="product_id", how="left")

denominator = order_items_size[order_items_size["size"].isin(size_choices)].groupby("size").size()
numerator = returns_size[returns_size["size"].isin(size_choices)].groupby("size").size()

q9_rate = (numerator / denominator).sort_values(ascending=False)
q9_top_size = q9_rate.index[0]
choices_q9 = {"A": "S", "B": "M", "C": "L", "D": "XL"}
ans_q9 = next((k for k, v in choices_q9.items() if v == q9_top_size), "N/A")

print("Q9 return rate by size:")
print(q9_rate)
print(f"Top size: {q9_top_size}")
print(f"Option: {ans_q9}) {q9_top_size}")

Q9 return rate by size:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64
Top size: S
Option: A) S


In [11]:
# Q10: Installment plan with highest average payment value per order
q10_order_payment = payments.groupby(["order_id", "installments"], as_index=False)["payment_value"].sum()
installment_avg = q10_order_payment.groupby("installments")["payment_value"].mean().sort_values(ascending=False)

top_installments = int(installment_avg.index[0])
choices_q10 = {"A": 1, "B": 3, "C": 6, "D": 12}
ans_q10 = next((k for k, v in choices_q10.items() if v == top_installments), "N/A")

print("Q10 avg payment_value per order by installments:")
print(installment_avg)
print(f"Top installments: {top_installments}")
print(f"Option: {ans_q10}) {top_installments} kỳ")

Q10 avg payment_value per order by installments:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
Top installments: 6
Option: C) 6 kỳ
